# Lesson 6
#### Classes and OOP (Object Oriented Programming)
- Dunder functions (double underscore)
- Type hinting

In [ ]:
class Point:
    def __init__(self, x: int, y: int): #  type hinting when indicating which variable type you want
        self.x = x
        self.y = y

    def __repr__(self):  # representation function, should describe the object and type
        cls = type(self).__name__
        return f"{cls}(x={self.x}, y={self.y})"

    def __str__(self):   # this is the text when using print() function on object
        return f"({self.x}, {self.y})"

    def __eq__(self, other):  # equals to function, how the operator == should act with a object of the class
        if not isinstance(other, Point):
            return NotImplemented
        return self.x == other.x and self.y == other.y

    def distance_to_origin(self) -> float:  # -> is also type hinting, what the function returns
        return(self.x**2 + self.y**2) ** 0.5

p1 = Point(5, 6)
print("p1 avstånd till origo", p1.distance_to_origin())
print(repr(p1))
print(p1)

## Encapsulation & @property
- for internal (convention): _internal_name
- Name mangling: __name -> changes name to _Classname__name -> "reduces the risk of name collisions in inheritance

In [ ]:
# same class as above
class Point:
    def __init__(self, x: int, y: int):
        self.x = x
        self.y = y

    def __str__(self): 
        return f"({self.x}, {self.y})"

    @property
    def magnitude(self) -> float:
        # calculated field, looks like a attribute when used
        return(self.x**2 + self.y**2) ** 0.5

p = Point(3, 4)
print(p.magnitude)  # prints like a variable instead of function call

In [ ]:
class Account:
    def __init__(self, owner: str, balance: float = 0.0):
        self.owner = owner
        # internal field signals convention "do not touch from outside the class"
        self._balance = 0.0
        # use property to get validation also in init
        self.balance = balance

    @property
    def balance(self):
        return self._balance

    @balance.setter
    def balance(self, value: float):
        if value < 0:
            raise ValueError("Balance may not be negative.")
        self._balance = float(value)

    def deposit(self, amount: float):
        if amount <= 0:
            raise ValueError("Deposit must be positive.")
        self.balance = self._balance + amount

    def withdraw(self, amount: float):
        if amount <= 0:
            raise ValueError("Withdrawal must be positive.")
        self.balance = self._balance - amount

a = Account("Håkan", 110)
print (a.balance)
a.withdraw(50)
print (a.balance)


Name mangling

In [ ]:
class Demo:
    def __init__(self):
        self.__token = "secret"

    def reveal(self):
        return self.__token

d = Demo()
print(d.reveal())
# print(d.__token) # will get an error since __token will be hidden outside object


## Composition

In [ ]:
class Item:
    def __init__(self, name: str, price: float, quantity: int = 1):
        self.name = name
        self.price = price
        self.quantity = quantity

    def __repr__(self):
        return f"Item(name={self.name!r}, price={self.price}, quantity={self.quantity})"

    @property
    def line_total(self):
        return self.price * self.quantity

i1 = Item("Banan", 5.0, 3)

print(i1)
print(i1.line_total)

In [ ]:
from typing import List  # for typehinting list with objects

class Cart:
    def __init__(self):
        self._items: List[Item] = []

    def add(self, item: Item):
        for existing in self._items:
            if existing.name == item.name and existing.price == item.price:
                existing.quantity += item.quantity
                return
        self._items.append(item)

    def remove_by_name(self, name: str):
        for idx, it in enumerate(self._items):
            if it.name == name:
                self._items.pop(idx)
                return True
        return False

    @property
    def subtotal(self):
        return sum(it.line_total for it in self._items) #list comprehension (loop as an argument)

cart = Cart()
cart.add(Item("Banan", 5.0, 3))
cart.add(Item("Äpple", 6.0, 2))
cart.add(Item("Banan", 5.0, 3))

print(cart._items)
cart.remove_by_name("Äpple")
print(cart._items)
print(cart.subtotal)

## Inheritance

In [ ]:
class Shape:
    def __init__(self, name: str, color: str = "black"):
        self.name = name
        self.color = color

    def area(self):
        raise NotImplementedError("Subclasses must implement .area()")

    def describe(self):
        # :.2f means (:) starts the format rules (.2) means 2 decimals and
        # (f) means float  
        return f"{self.name} (color = {self.color}) has area {self.area():.2f}"

    def  __repr__(self):
        return f"Shape(name={self.name!r}, color={self.color!r})"

In [ ]:
import math
# subclass
class Circle(Shape):
    def __init__(self, radius: float, color: str = "black"):
        super().__init__(name = "Circle", color = color)
        self.radius = radius

    def area(self):
        return math.pi * (self.radius ** 2)

    def __repr__(self):
        return f"Circe(radius={self.radius}, color={self.color!r})"

class Rectangle(Shape):
    def __init__(self, width: float, height: float, color: str = "black"):
        super().__init__(name = "Rectangle", color = color)
        self.width = width
        self.height = height

    def area(self):
        return self.width * self.height

    def __repr__(self):
        return f"Rectangle(width={self.width}, height={self.height}, color={self.color!r})"

c = Circle(3, color = "red")
r = Rectangle(4,2, color = "blue")
print(c)
print(r)
print(c.describe())


## Dataclasses
- a special type of class that is used primarily to store data instead of holding complex   business logic
- The main benefit is that Python automatically writes common dunder methods for you,  
 such as \_\_init\_\_, \_\_repr\_\_, and \_\_eq\_\_, based on the variables (type hints) you define.

In [32]:
# example regular class
class Item2:
    def __init__(self, name: str, price: float, quantity: int = 1):
        self.name = name
        self.price = float(price)
        self.quantity = int(quantity)

    def __repr__(self):
        return f"Item(name={self.name!r}, price={self.price}, quantity={self.quantity})"

    def __eq__(self, other: object):
        if not isinstance(other, Item2):
            return NotImplemented
        return (self.name, self.price, self.quantity) == (other.name, other.price, other.quantity)

i1 = Item2("Banan", 5.0, 3)
i2 = Item2("Banan", 5.0, 3)
print(i1)
print(i1 == i2)

Item(name='Banan', price=5.0, quantity=3)
True


In [31]:
from dataclasses import dataclass

# same class as dataclass
@dataclass
class Item3:
    name: str
    price: float
    quantity: int = 1

i1 = Item3("Banan", 5.0, 3)
i2 = Item3("Banan", 5.0, 3)
print(i1)
print(i1 == i2)

Item3(name='Banan', price=5.0, quantity=3)
True


- frozen state example
- if you need to have a init, write as def \_\_post_init\_\_(self)

In [40]:
from dataclasses import dataclass, field

@dataclass(frozen=True) 
# locks the values to be changed with command like o1.name = "Stefan"
# will raise a FrozenInstanceError
class Order:
    customer: str
    items: List[Item3] = field(default_factory=list)
    # this needs to be added if list, set, dictionary or tuple to avoid 
    # sharing between instances of the class

o1 = Order("Håkan")
o1.items.append(Item3("Banan", 5.0))
